# Assignment 2: Improving IMDB Sentiment Classification

This notebook extends Assignment 1 by training and evaluating multiple models on the IMDB movie review dataset.

## Models included
1. **Tuned MLP with lexicon features** (VADER + TextBlob)
2. **LSTM**
3. **3. 1D CNN with trainable embedding layer**

## Techniques applied
- text cleaning and preprocessing
- hyperparameter tuning / architecture improvements
- dropout regularization
- early stopping
- padded input sequences for neural models
- trainable embedding layer for sequence representation

Each model saves:
- a trained model file
- a separate `results.json`
- a separate `README.md`

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib

In [ ]:
# !pip install nltk textblob gensim tensorflow scikit-learn joblib

import nltk
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer

from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, LSTM, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Change this path if your CSV is in a data/ folder
DATA_PATH = "IMDB Dataset.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
def clean_review(x):
    if not isinstance(x, str):
        return ""
    x = x.lower()
    x = re.sub(r"<br\s*/?>", " ", x)
    x = re.sub(r"[^a-z0-9\s']", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["review_clean"] = df["review"].astype(str).apply(clean_review)
df["y"] = df["sentiment"].str.lower().map({"positive": 1, "negative": 0})

print(df["y"].value_counts())
df[["review_clean", "y"]].head()

In [ ]:
train_df, test_df = train_test_split(
    df[["review_clean", "y"]],
    test_size=0.2,
    random_state=SEED,
    stratify=df["y"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["y"]
)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

In [ ]:
os.makedirs("assignment2", exist_ok=True)

def save_results_and_readme(model_dir, metrics, readme_text):
    os.makedirs(model_dir, exist_ok=True)

    with open(os.path.join(model_dir, "results.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    with open(os.path.join(model_dir, "README.md"), "w") as f:
        f.write(readme_text)

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_prob))
    }

In [ ]:
#model 1: tuning MLP with VADER + TextBlob features (larger architecture and cleaner preprocessing)
def get_vader_feats(text):
    s = sia.polarity_scores(text)
    return s["neg"], s["neu"], s["pos"], s["compound"]

def get_textblob_feats(text):
    t = TextBlob(text).sentiment
    return float(t.polarity), float(t.subjectivity)

def build_lexicon_features(text_series):
    vader_feats = text_series.apply(get_vader_feats)
    textblob_feats = text_series.apply(get_textblob_feats)

    feats = pd.DataFrame({
        "tb_pol": textblob_feats.apply(lambda x: x[0]),
        "tb_subj": textblob_feats.apply(lambda x: x[1]),
        "vader_neg": vader_feats.apply(lambda x: x[0]),
        "vader_neu": vader_feats.apply(lambda x: x[1]),
        "vader_pos": vader_feats.apply(lambda x: x[2]),
        "vader_comp": vader_feats.apply(lambda x: x[3]),
    })
    return feats

X_train_lex = build_lexicon_features(train_df["review_clean"])
X_val_lex = build_lexicon_features(val_df["review_clean"])
X_test_lex = build_lexicon_features(test_df["review_clean"])

y_train = train_df["y"].values
y_val = val_df["y"].values
y_test = test_df["y"].values

X_train_lex.head()

In [ ]:
mlp_lex = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=200,
    random_state=SEED,
    verbose=False
)

model_lex = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", mlp_lex)
])

model_lex.fit(X_train_lex, y_train)

pred_lex = model_lex.predict(X_test_lex)
prob_lex = model_lex.predict_proba(X_test_lex)[:, 1]

metrics_lex = evaluate_predictions(y_test, pred_lex, prob_lex)
metrics_lex

In [ ]:
print("Tuned lexicon MLP")
print(metrics_lex)
print("\nClassification report:\n")
print(classification_report(y_test, pred_lex, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_lex))

model_dir = "assignment2/mlp_lexicon_tuned"
joblib.dump(model_lex, os.path.join(model_dir, "model.joblib"))

readme_lex = """# Tuned MLP with Lexicon Features

## Model architecture
- Input features: 6 sentiment-based features
  - TextBlob polarity
  - TextBlob subjectivity
  - VADER neg / neu / pos / compound
- StandardScaler
- MLPClassifier with hidden layers (64, 32)
- ReLU activation
- Adam optimizer

## Techniques applied
- text cleaning
- lexicon feature engineering
- scaling
- hyperparameter tuning (larger hidden layers and more iterations than Assignment 1)
- evaluation with accuracy, precision, recall, F1, and ROC-AUC
"""

save_results_and_readme(model_dir, metrics_lex, readme_lex)

In [ ]:
#Model 2: LSTM (tokenized review sequences instead of only handcrafted sentiment scores)
MAX_WORDS = 10000
MAX_LEN = 200
EMBED_DIM = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["review_clean"])

X_train_seq = tokenizer.texts_to_sequences(train_df["review_clean"])
X_val_seq = tokenizer.texts_to_sequences(val_df["review_clean"])
X_test_seq = tokenizer.texts_to_sequences(test_df["review_clean"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print(X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
prob_lstm = lstm_model.predict(X_test_pad).ravel()
pred_lstm = (prob_lstm >= 0.5).astype(int)

metrics_lstm = evaluate_predictions(y_test, pred_lstm, prob_lstm)
metrics_lstm

In [ ]:
print("LSTM")
print(metrics_lstm)
print("\nClassification report:\n")
print(classification_report(y_test, pred_lstm, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_lstm))

model_dir = "assignment2/lstm"
lstm_model.save(os.path.join(model_dir, "model.keras"))

readme_lstm = """# LSTM for IMDB Sentiment Classification

## Model architecture
- Tokenizer vocabulary size: 10000
- Sequence length: 200
- Embedding layer: 100 dimensions
- LSTM with 64 units
- Dense hidden layer with 64 units and ReLU
- Dropout 0.5
- Sigmoid output layer

## Techniques applied
- text cleaning
- tokenization and sequence padding
- recurrent neural network (LSTM)
- dropout regularization
- early stopping
- evaluation with accuracy, precision, recall, F1, and ROC-AUC
"""

save_results_and_readme(model_dir, metrics_lstm, readme_lstm)

In [ ]:
#Model3: 1D CNN with Word2Vec embeddings(Word2Vec is trained on the IMDB training reviews, and the learned vectors are used to initialize the embedding layer.)

sentences_for_w2v = [text.split() for text in train_df["review_clean"]]

w2v_model = Word2Vec(
    sentences=sentences_for_w2v,
    vector_size=EMBED_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=SEED
)

word_index = tokenizer.word_index
num_words = min(MAX_WORDS, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, EMBED_DIM))

for word, i in word_index.items():
    if i >= MAX_WORDS:
        continue
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

In [ ]:
cnn_model = Sequential([
    Embedding(
        input_dim=num_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False
    ),
    Conv1D(filters=128, kernel_size=5, activation="relu"),
    GlobalMaxPooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_cnn = cnn_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
prob_cnn = cnn_model.predict(X_test_pad).ravel()
pred_cnn = (prob_cnn >= 0.5).astype(int)

metrics_cnn = evaluate_predictions(y_test, pred_cnn, prob_cnn)
metrics_cnn

In [ ]:
print("1D CNN with Word2Vec embeddings")
print(metrics_cnn)
print("\nClassification report:\n")
print(classification_report(y_test, pred_cnn, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_cnn))

model_dir = "assignment2/cnn_word2vec"
cnn_model.save(os.path.join(model_dir, "model.keras"))

readme_cnn = """# 1D CNN with Word2Vec Embeddings

## Model architecture
- Tokenizer vocabulary size: 10000
- Sequence length: 200
- Word2Vec embedding matrix with 100 dimensions
- Embedding layer initialized with Word2Vec weights
- Conv1D with 128 filters and kernel size 5
- GlobalMaxPooling1D
- Dense hidden layer with 64 units and ReLU
- Dropout 0.5
- Sigmoid output layer

## Techniques applied
- text cleaning
- tokenization and padding
- pretrained word embeddings using Gensim Word2Vec trained on the IMDB training set
- 1D CNN for text classification
- dropout regularization
- early stopping
- evaluation with accuracy, precision, recall, F1, and ROC-AUC
"""

save_results_and_readme(model_dir, metrics_cnn, readme_cnn)

In [ ]:
#Comparing model performance
comparison = pd.DataFrame([
    {"model": "Tuned MLP (lexicon)", **metrics_lex},
    {"model": "LSTM", **metrics_lstm},
    {"model": "1D CNN + Word2Vec", **metrics_cnn},
])

comparison = comparison.sort_values("accuracy", ascending=False).reset_index(drop=True)
comparison

In [ ]:
comparison.to_csv("assignment2/model_comparison.csv", index=False)

comparison_md = """# Assignment 2 Model Comparison

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|------|----------|-----------|--------|----|---------|
"""

for _, row in comparison.iterrows():
    comparison_md += f"| {row['model']} | {row['accuracy']:.4f} | {row['precision']:.4f} | {row['recall']:.4f} | {row['f1']:.4f} | {row['roc_auc']:.4f} |\n"

with open("assignment2/comparison.md", "w") as f:
    f.write(comparison_md)

comparison